# Muon vs SGD：分阶段线性网络

两层线性网络 $\hat y = W_2 W_1 x$，目标 $W_2 W_1 \approx A$。

**目标分解** 对 $A$ 做 SVD：$A = U \Sigma V^\top$，定义
$$W_1^* = \sqrt{\Sigma}\, V^\top, \quad W_2^* = U \sqrt{\Sigma}$$
则 $W_2^* W_1^* = A$。Phase 1 只训练 $W_1$，以 $\|W_1 - W_1^*\|$ 为损失；Phase 2 只训练 $W_2$，以合成误差 $\|W_2 W_1 - A\|$ 为损失。

**分阶段训练**
- Phase 1（比较 SGD 与 Muon）：冻结 $W_2$，用 SGD 或 Muon 训 $W_1$，最小化 $\|W_1 - W_1^*\|^2 / \|A\|^2$
- Phase 2（固定 SGD）：冻结 Phase 1 得到的 $W_1$，重初始化 $W_2$，仅用 SGD 最小化 $\|W_2 W_1 - A\|^2 / \|A\|^2$，直到 $\|W_2 W_1 - A\| / \|A\| \le \text{THRESHOLD}$

Muon：与 SGD 相同 lr；每步对当前梯度做 NS 正交化，更新量范数为 lr $\|g\|$。

In [ ]:
import math

import torch

torch.set_default_dtype(torch.float64)

D = 64
PHASE1_LR = 2e-1
PHASE2_LR = 2e-2
SEED = 0
THRESHOLD = 0.05
PHASE1_STEPS = 600
PHASE2_MAX_STEPS = 20000

In [ ]:
def zeropower_via_newtonschulz5(G, steps=3, eps=1e-7):
    assert len(G.shape) == 2
    a, b, c = (3.4445, -4.7750, 2.0315)
    X = G.bfloat16()
    X = X / (X.norm() + eps)
    if G.size(0) > G.size(1):
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * A @ A
        X = a * X + B @ X
    if G.size(0) > G.size(1):
        X = X.T
    return X.to(dtype=G.dtype)


def muon_direction(g, eps=1e-7):
    direction = zeropower_via_newtonschulz5(g)
    return direction * (g.norm() / (direction.norm() + eps))


class Muon(torch.optim.Optimizer):

    def __init__(self, params, lr=1e-3):
        super().__init__(params, dict(lr=lr))

    def step(self):
        for group in self.param_groups:
            lr = group["lr"]
            for p in group["params"]:
                g = p.grad
                if g is None:
                    continue
                p.data.add_(muon_direction(g), alpha=-lr)

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_target(d, seed=0):
    gen = torch.Generator().manual_seed(seed)
    return torch.randn(d, d, generator=gen) / math.sqrt(d)


def svd_factors(A):
    u, s, vh = torch.linalg.svd(A.cpu(), full_matrices=False)
    sqrt_s = torch.sqrt(s)
    w1_star = (torch.diag(sqrt_s) @ vh).cuda()
    w2_star = (u @ torch.diag(sqrt_s)).cuda()
    return w1_star, w2_star


def rel_dist(M, target, scale):
    return (M - target).norm().item() / (scale + 1e-12)


def compose_loss(W2, W1, A, norm_A):
    M = W2 @ W1 - A
    return (M ** 2).sum() / (norm_A**2)


def steps_to(values, threshold):
    for i, v in enumerate(values):
        if v <= threshold:
            return i
    return None


def rand_matrix(d, seed):
    state = torch.cuda.get_rng_state()
    torch.cuda.manual_seed(seed)
    W = torch.randn(d, d, device="cuda") / math.sqrt(d)
    torch.cuda.set_rng_state(state)
    return W

In [ ]:
def train_staged(
    A,
    w1_star,
    w2_star,
    phase1_optimizer_class,
    phase1_steps=PHASE1_STEPS,
    phase2_max_steps=PHASE2_MAX_STEPS,
    phase2_threshold=THRESHOLD,
    phase2_lr=PHASE2_LR,
    seed=SEED,
):
    set_seed(seed)
    W1 = rand_matrix(D, seed)
    norm_A = A.norm().item()
    w1_star = w1_star.detach()

    w1_dist = []

    W1.requires_grad_(True)
    opt1 = phase1_optimizer_class([W1], lr=PHASE1_LR)
    for step in range(phase1_steps + 1):
        w1_dist.append(rel_dist(W1.detach(), w1_star, norm_A))
        if step == phase1_steps:
            break
        loss = ((W1 - w1_star) ** 2).sum() / (norm_A**2)
        opt1.zero_grad(set_to_none=True)
        loss.backward()
        opt1.step()

    W1_frozen = W1.detach().clone()

    W2 = rand_matrix(D, seed + 1)
    W2.requires_grad_(True)
    opt2 = torch.optim.SGD([W2], lr=phase2_lr)
    tot_dist, w2_dist = [], []
    phase2_steps = 0
    for step in range(phase2_max_steps + 1):
        W2d = W2.detach()
        dist = rel_dist(W2d @ W1_frozen, A, norm_A)
        tot_dist.append(dist)
        w2_dist.append(rel_dist(W2d, w2_star, norm_A))
        phase2_steps = step
        if dist <= phase2_threshold:
            break
        if step == phase2_max_steps:
            break
        loss = compose_loss(W2, W1_frozen, A, norm_A)
        opt2.zero_grad(set_to_none=True)
        loss.backward()
        opt2.step()

    return dict(
        tot_dist=tot_dist,
        w1_dist=w1_dist,
        w2_dist=w2_dist,
        W1=W1_frozen,
        W2=W2.detach(),
        phase1_steps=phase1_steps,
        phase2_steps=phase2_steps,
    )


def run_staged_experiment(A, w1_star, w2_star):
    print("\n" + "=" * 72)
    print(f"Staged training: phase1 ({PHASE1_STEPS} steps), phase2 (threshold {THRESHOLD})")
    print(f"  phase1 lr={PHASE1_LR}, loss = ||W1-W1*||^2/||A||^2")
    print(f"  phase2 lr={PHASE2_LR}, loss = ||W2 W1 - A||^2/||A||^2")
    print("=" * 72)
    sgd = train_staged(A, w1_star, w2_star, torch.optim.SGD)
    muon = train_staged(A, w1_star, w2_star, Muon)
    norm_A = A.norm().item()

    for label, run in [("SGD W1 -> SGD W2", sgd), ("Muon W1 -> SGD W2", muon)]:
        p1 = run["phase1_steps"]
        print(f"\n  [{label}]")
        print(f"    phase1 step {p1}: ||W1-W1*||={run['w1_dist'][-1]:.4f}")
        print(f"    phase2 step {p1 + run['phase2_steps']}: ||W1-W1*||={rel_dist(run['W1'], w1_star, norm_A):.4f}, "
              f"||W2-W2*||={run['w2_dist'][-1]:.4f}, ||W2 W1 - A||/||A||={run['tot_dist'][-1]:.6f}, "
              f"steps={run['phase2_steps']}")
        p2_hit = steps_to(run["tot_dist"], THRESHOLD)
        total = p1 + p2_hit if p2_hit is not None else None
        print(f"    steps@{THRESHOLD} total={total}")

    sgd_p2 = steps_to(sgd["tot_dist"], THRESHOLD)
    muon_p2 = steps_to(muon["tot_dist"], THRESHOLD)
    delta = sgd_p2 - muon_p2 if sgd_p2 and muon_p2 else None
    print(f"\n  delta={delta} (positive => Muon faster)")
    return sgd, muon

In [ ]:
set_seed(SEED)
A = make_target(D, seed=SEED).cuda()
w1_star, w2_star = svd_factors(A)
sgd_run, muon_run = run_staged_experiment(A, w1_star, w2_star)

## 小结

Muon Phase1 离 $W_1^*$ 更远，但 $W_1$ 谱更利于 Phase2。净效果：Phase2 步数可能更少，整体优化过程仍可能更快（`delta > 0`）。

为什么 Muon 反而在 Phase 2 更快？

Phase 2 等价于固定编码 $W_1$、学线性探测 $W_2$ 使 $W_2 W_1 \approx A$。令 $E_t = W_{2,t}W_1 - A$，则
$$L(W_2)=\tfrac12\|W_2 W_1 - A\|_F^2,\qquad \nabla_{W_2}L=(W_2 W_1 - A)W_1^\top$$
SGD 更新 $W_{2,t+1}=W_{2,t}-\eta(W_{2,t}W_1-A)W_1^\top$，误差递推为
$$E_{t+1}=E_t(I-\eta W_1^\top W_1)$$
在 $W_1^\top W_1$ 的特征基下，$E_t$ 的第 $i$ 个奇异值 $s_i(E_t)$ 满足 $s_i(E_{t+1})=|1-\eta\sigma_i^2|\,s_i(E_t)$，其中 $\sigma_i$ 是 $W_1$ 的第 $i$ 个奇异值。取 $\eta=\alpha/\sigma_{\max}(W_1)^2$ 时，最慢收缩因子为 $1-\alpha/\kappa(W_1)^2$。

因此，$W_1$ 越接近正交（谱越平、条件数越小），Phase 2 的优化越快。

上文直接用 $W_2 W_1 - A$ 构造损失；实践中更常见的做法是给定样本 $x$，计算输出 $y=W_2 W_1 x$，再比较 $y$ 与目标 $Ax$。

可以证明，$W_1$ 越接近正交，中间层表征 $h=W_1 x$ 经线性探测恢复原始输入 $x$ 就越容易——这与前面说的学习 $Ax$ 更容易是同样的道理。恢复 $x$ 时学探测 $B$ 使 $B W_1 \approx I$，损失为
$$L_x(B)=\tfrac12\|B W_1 - I\|_F^2$$
恢复 $Ax$ 时只需把目标矩阵换成 $A$：
$$L_A(B)=\tfrac12\|B W_1 - A\|_F^2$$
令 $E_t = B_t W_1 - M$（$M=I$ 或 $M=A$），则仍有 $E_{t+1} = E_t(I - \eta W_1^\top W_1)$，收敛快慢同样只由 $W_1$ 的奇异值决定。

## 对 Muon 的理解

综上所述，Muon 的优势可以这样理解：上游每次更新同时承担两种角色——一是沿负梯度方向降低当前损失，让训练继续；二是像临时的残差连接，尽量把输入信息原样编码进输出、传递给下游。因为下游究竟更需要 $x$ 还是 $W_1 x$ 事先并不清楚，用正交变换编码 $x$ 是最利于下游线性恢复 $x$ 的做法。Muon 的权衡，正是把梯度正交化后用作更新方向：对 Phase 1 这个「子问题」而言，它不再是该损失下的最速下降；但由此得到的 $W_1$ 为下游提供了更好的表征，Phase 2 反而更容易，而 Phase 2 优化得更好，反过来又可以给 Phase 1 提供更容易的「子问题」，弥补了在 Phase 1 里 Muon 「解题能力」 不如 SGD 的弱点，最终让 Muon 的整体优化反而更快。